<a href="https://colab.research.google.com/github/khushi2460/ML-Projects/blob/main/Realesatechatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import spacy
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import gradio as gr

# Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/Giridharan79/HOUSE_PRICE_PREDICTION/main/final_dataset.csv")

# Rename column from zip_code → pin_code
df = df.rename(columns={"zip_code": "pin_code"})

# Features and target
X = df[['beds', 'baths', 'size', 'pin_code']]
y = df['price']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Load NLP model
nlp = spacy.load("en_core_web_sm")

def _check_keywords_in_window(doc, current_index, keywords, window_size=2):
    """Checks for presence of any keyword within a specified window around current_index."""
    for k in range(max(0, current_index - window_size), min(len(doc), current_index + window_size + 1)):
        if k == current_index: # Skip the number token itself
            continue
        if any(kw.lower() in doc[k].text.lower() for kw in keywords):
            return True
    return False

# Chatbot function
def real_estate_chatbot(message, history=[]):
    doc = nlp(message.lower())

    beds_val, baths_val, size_val, pin_code_val = None, None, None, None

    # Iterate through tokens to find numbers and their contexts
    for i, token in enumerate(doc):
        if token.like_num:
            num = int(token.text)

            # Check for beds (e.g., '2 bhk', '3 beds', '4 bedroom', '3 bedrooms')
            if beds_val is None and _check_keywords_in_window(doc, i, ["bhk", "bed", "bedroom", "bedrooms"]):
                beds_val = num
            # Check for baths (e.g., '2 baths', '3 bathroom', '4 bathrooms')
            elif baths_val is None and _check_keywords_in_window(doc, i, ["bath", "bathroom", "bathrooms"]):
                baths_val = num
            # Check for size (e.g., '1200 sq ft', '500 square size', '1200 sqft', '1200 sft')
            elif size_val is None and _check_keywords_in_window(doc, i, ["sq", "size", "square", "ft", "feet", "sqft", "sft"]):
                size_val = num
            # Check for pin code / zip code
            # Prioritize 'pin'/'zip' keyword, then check if it's a standalone 5-digit number
            elif pin_code_val is None:
                if _check_keywords_in_window(doc, i, ["pin", "pincode", "zip", "zipcode"]):
                    pin_code_val = num
                elif len(token.text) == 5 and token.text.isdigit():
                    pin_code_val = num

    if beds_val is not None and baths_val is not None and size_val is not None and pin_code_val is not None:
        input_data = pd.DataFrame([[beds_val, baths_val, size_val, pin_code_val]], columns=X.columns)
        prediction = model.predict(input_data)[0]
        # Assuming prices are in USD based on the dataset (Seattle, Washington)
        return f"Estimated Price: ${prediction:,.0f}"
    else:
        missing_info = []
        if beds_val is None:
            missing_info.append("number of beds (e.g., '2BHK' or '2 beds')")
        if baths_val is None:
            missing_info.append("number of baths (e.g., '3 baths')")
        if size_val is None:
            missing_info.append("size in square feet (e.g., '1200 sq ft')")
        if pin_code_val is None:
            missing_info.append("PIN code or Zip Code (e.g., 'PIN 98101' or 'ZIP 98101')")

        if missing_info:
            return f"I need more information to estimate the price. Please provide the {', '.join(missing_info)}."
        else: # Fallback, should ideally not be reached if all missing_info checks are present
            return "Please mention details like beds, baths, size (sq ft), and PIN code."

# Gradio chatbot interface
gr.ChatInterface(
    fn=real_estate_chatbot,
    title="🏠 Real Estate Chatbot",
    description="Ask about house prices in natural language. Example: 'Find me a 2-bedroom house with 3 bathrooms, 1200 square feet, and zip code 98101'"
).launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f06de56cd18d5ae99.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
